# T1.7 — Retrospective Power / Replicate-Count Justification

**Analysis-only. No new models are trained.** This notebook uses the locked 30-model primary endpoint table (6 rho levels × 5 replicates).

## Purpose
- Estimate retrospective power at the completed **n=5 replicates per rho** design.
- Estimate the **minimum detectable effect (MDE)** at **n=3 replicates per rho**, matching the reduced-replicate Tier 2/3 designs.
- Use the same association framework as the locked primary analysis: **Spearman rank correlation between rho and endpoint**.

## Statistical approximation
The locked primary analysis uses a permutation-Spearman test. A full permutation-of-permutations calculation inside 20,000 power simulations would be unnecessarily expensive. Therefore, the simulation below uses the **parametric Spearman p-value** (`scipy.stats.spearmanr`) as a computational approximation to the primary permutation-Spearman test.

**Important:** this is a retrospective design-sensitivity analysis, not proof that the observed effect was guaranteed to be detectable before data collection. The n=3 MDE is conditional on the variability estimated from the completed primary experiment.

If any endpoint's simulated power is close to the decision boundary of **80%**, the notebook flags it for a smaller true-permutation-Spearman confirmation analysis.


In [1]:
# CELL 1 — locked settings
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

SEED = 20260924
ALPHA = 0.05
TARGET_POWER = 0.80
BOUNDARY_TOL = 0.05       # flag approximately 75–85% power
N_SIMS = 20_000
N_CONFIRM = 2_000         # only used for optional true-permutation confirmation
RHO_LEVELS = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
N_PRIMARY = 5
N_REDUCED = 3
ENDPOINTS = ["delta_z_hair", "D", "D_AUROC", "xai_enrichment"]

print("T1.7 — RETROSPECTIVE POWER / REPLICATE-COUNT JUSTIFICATION")
print(f"alpha={ALPHA}, target power={TARGET_POWER:.0%}, simulations={N_SIMS:,}, seed={SEED}")
print("Primary-test approximation: parametric Spearman p-value")


T1.7 — RETROSPECTIVE POWER / REPLICATE-COUNT JUSTIFICATION
alpha=0.05, target power=80%, simulations=20,000, seed=20260924
Primary-test approximation: parametric Spearman p-value


In [4]:
# CELL 2 — load and validate the frozen 30-model primary endpoint table

from pathlib import Path
import numpy as np
import pandas as pd

TARGET_FILE = "model_endpoints_final_checkpoint.csv"

# Search standard Kaggle locations recursively
search_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

matches = []

for root in search_roots:
    if root.exists():
        matches.extend(root.rglob(TARGET_FILE))

# Remove duplicates while preserving order
matches = list(dict.fromkeys(matches))

if not matches:
    raise FileNotFoundError(
        f"{TARGET_FILE} was not found anywhere under /kaggle/input or /kaggle/working.\n\n"
        "This file must be restored from the completed primary Stage-4 analysis."
    )

if len(matches) > 1:
    print("Multiple copies found:")
    for p in matches:
        print(" -", p)

    # Prefer a working copy, otherwise first input copy
    working = [p for p in matches if str(p).startswith("/kaggle/working")]
    CSV_PATH = working[0] if working else matches[0]
else:
    CSV_PATH = matches[0]

print("Using frozen endpoint table:")
print(CSV_PATH)

df = pd.read_csv(CSV_PATH)

ENDPOINTS = [
    "delta_z_hair",
    "D",
    "D_AUROC",
    "xai_enrichment"
]

required = {
    "model",
    "rho",
    "replicate",
    "checkpoint",
    *ENDPOINTS
}

missing = sorted(required - set(df.columns))

if missing:
    raise ValueError(
        f"Frozen endpoint table is missing required columns: {missing}"
    )

# Primary-design integrity checks
assert len(df) == 30, f"Expected 30 primary models, got {len(df)}"
assert df["model"].nunique() == 30
assert df["rho"].nunique() == 6

counts = df.groupby("rho").size().sort_index()
print("\nReplicate counts by rho:")
print(counts)

assert all(counts == 5), "Primary design must have exactly 5 replicates per rho."

RHO_LEVELS = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

assert np.all(
    np.sort(df["rho"].unique()) == RHO_LEVELS
), "Unexpected rho levels."

for endpoint in ENDPOINTS:
    values = df[endpoint].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError(
            f"Non-finite values found in endpoint: {endpoint}"
        )

print("\n✓ Frozen 30-model endpoint table validated.")
print(f"✓ Rows: {len(df)}")
print(f"✓ Models: {df['model'].nunique()}")
print(f"✓ rho levels: {sorted(df['rho'].unique())}")
print("✓ 5 replicates at every rho")
print("✓ All required endpoints finite")

Using frozen endpoint table:
/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/results/model_endpoints_final_checkpoint.csv

Replicate counts by rho:
rho
0.0    5
0.2    5
0.4    5
0.6    5
0.8    5
1.0    5
dtype: int64

✓ Frozen 30-model endpoint table validated.
✓ Rows: 30
✓ Models: 30
✓ rho levels: [np.float64(0.0), np.float64(0.2), np.float64(0.4), np.float64(0.6), np.float64(0.8), np.float64(1.0)]
✓ 5 replicates at every rho
✓ All required endpoints finite


In [5]:
# CELL 3 — observed Spearman effects and residual-noise estimates

observed = []

for ep in ENDPOINTS:
    x = df["rho"].to_numpy(float)
    y = df[ep].to_numpy(float)

    sp = stats.spearmanr(x, y)
    fit = stats.linregress(x, y)

    # Noise scale estimated from the completed primary experiment.
    resid = y - (fit.intercept + fit.slope * x)
    residual_sd = float(np.sqrt(np.sum(resid**2) / (len(y) - 2)))

    observed.append({
        "endpoint": ep,
        "spearman_rho": float(sp.statistic),
        "parametric_spearman_p": float(sp.pvalue),
        "linear_slope": float(fit.slope),
        "linear_r2": float(fit.rvalue**2),
        "residual_sd": residual_sd,
    })

observed_df = pd.DataFrame(observed)
display(observed_df)


,endpoint,spearman_rho,parametric_spearman_p,linear_slope,linear_r2,residual_sd
0,delta_z_hair,0.986562,1.413467e-23,9.849509,0.978745,0.513180
1,D,0.986562,1.413467e-23,0.174170,0.963369,0.012008
2,D_AUROC,0.986562,1.413467e-23,0.357482,0.978095,0.018914
3,xai_enrichment,0.984307,1.223133e-22,9.073338,0.939669,0.812837


In [6]:
# CELL 4 — Monte Carlo power using the parametric Spearman p-value

def power_for_slope(slope, residual_sd, n_rep, rng, n_sims=N_SIMS):
    x = np.repeat(RHO_LEVELS, n_rep)
    signal = slope * x
    rejections = 0

    for _ in range(n_sims):
        y = signal + rng.normal(0.0, residual_sd, size=x.size)
        p = stats.spearmanr(x, y).pvalue
        if np.isfinite(p) and p < ALPHA:
            rejections += 1

    return rejections / n_sims


power_rows = []

for row in observed_df.itertuples(index=False):
    ep_index = ENDPOINTS.index(row.endpoint)

    for n_rep, design_name in [
        (N_PRIMARY, "n=5 primary"),
        (N_REDUCED, "n=3 reduced")
    ]:
        rng = np.random.default_rng(SEED + n_rep + ep_index * 1000)

        power = power_for_slope(
            row.linear_slope,
            row.residual_sd,
            n_rep,
            rng,
            N_SIMS
        )

        power_rows.append({
            "endpoint": row.endpoint,
            "design": design_name,
            "n_replicates_per_rho": n_rep,
            "observed_linear_slope": row.linear_slope,
            "residual_sd": row.residual_sd,
            "simulated_power": power,
            "target_power": TARGET_POWER,
            "boundary_flag_75_85pct": abs(power - TARGET_POWER) <= BOUNDARY_TOL,
            "test": "Spearman correlation with parametric p-value",
        })

power_df = pd.DataFrame(power_rows)
display(power_df)


,endpoint,design,n_replicates_per_rho,observed_linear_slope,residual_sd,simulated_power,target_power,boundary_flag_75_85pct,test
0,delta_z_hair,n=5 primary,5,9.849509,0.513180,1.0,0.8,False,Spearman correlation with parametric p-value
1,delta_z_hair,n=3 reduced,3,9.849509,0.513180,1.0,0.8,False,Spearman correlation with parametric p-value
2,D,n=5 primary,5,0.174170,0.012008,1.0,0.8,False,Spearman correlation with parametric p-value
3,D,n=3 reduced,3,0.174170,0.012008,1.0,0.8,False,Spearman correlation with parametric p-value
4,D_AUROC,n=5 primary,5,0.357482,0.018914,1.0,0.8,False,Spearman correlation with parametric p-value
5,D_AUROC,n=3 reduced,3,0.357482,0.018914,1.0,0.8,False,Spearman correlation with parametric p-value
6,xai_enrichment,n=5 primary,5,9.073338,0.812837,1.0,0.8,False,Spearman correlation with parametric p-value
7,xai_enrichment,n=3 reduced,3,9.073338,0.812837,1.0,0.8,False,Spearman correlation with parametric p-value


In [8]:
# CELL 5 — FAST MDE at n=3 for 80% power
#
# Same statistical specification:
#   - n = 3 replicates per rho
#   - target power = 80%
#   - parametric Spearman p-value
#   - 20,000 simulations
#
# This version is vectorized/batched so it does not call
# scipy.stats.spearmanr() 20,000 times for every candidate slope.

from scipy.stats import rankdata, t as student_t


def vectorized_spearman_power(
    slope,
    residual_sd,
    n_rep,
    rng,
    n_sims=N_SIMS,
    batch_size=1000
):
    x = np.repeat(RHO_LEVELS, n_rep).astype(float)
    n = len(x)

    # Fixed ranks of rho.
    rx = rankdata(x)
    rx_centered = rx - rx.mean()
    rx_norm = np.sqrt(np.sum(rx_centered ** 2))

    rejections = 0
    remaining = n_sims

    while remaining > 0:

        b = min(batch_size, remaining)

        # Simulated endpoint values
        y = (
            slope * x[None, :]
            + rng.normal(
                0.0,
                residual_sd,
                size=(b, n)
            )
        )

        # Rank each simulated row.
        order = np.argsort(
            y,
            axis=1,
            kind="quicksort"
        )

        ry = np.empty_like(
            order,
            dtype=np.float64
        )

        rows = np.arange(b)[:, None]
        ranks = np.arange(
            1,
            n + 1,
            dtype=np.float64
        )[None, :]

        ry[rows, order] = ranks
        ry -= ry.mean(axis=1, keepdims=True)

        # Spearman correlation
        numerator = ry @ rx_centered
        ry_norm = np.sqrt(
            np.sum(ry ** 2, axis=1)
        )

        r = numerator / (rx_norm * ry_norm)
        r = np.clip(r, -1.0, 1.0)

        # Parametric Spearman p-value
        df_t = n - 2

        denom = np.maximum(
            1.0 - r * r,
            np.finfo(float).eps
        )

        t_stat = (
            r
            * np.sqrt(df_t / denom)
        )

        p = 2.0 * student_t.sf(
            np.abs(t_stat),
            df_t
        )

        rejections += int(
            np.count_nonzero(p < ALPHA)
        )

        remaining -= b

    return rejections / n_sims


def estimate_mde_fast(
    residual_sd,
    n_rep=N_REDUCED,
    target_power=TARGET_POWER,
    seed=SEED
):

    # --------------------------------------------------------------
    # 1. Find an upper bracket where power >= 80%
    # --------------------------------------------------------------

    lo = 0.0
    hi = max(
        2.0 * residual_sd,
        1e-12
    )

    bracket_found = False

    for k in range(8):

        rng = np.random.default_rng(
            seed + 100 + k
        )

        p_hi = vectorized_spearman_power(
            hi,
            residual_sd,
            n_rep,
            rng,
            N_SIMS
        )

        if p_hi >= target_power:
            bracket_found = True
            break

        hi *= 2.0

    if not bracket_found:
        return np.nan, np.nan

    # --------------------------------------------------------------
    # 2. Bisection to find the minimum detectable slope
    # --------------------------------------------------------------

    for k in range(10):

        mid = 0.5 * (lo + hi)

        rng = np.random.default_rng(
            seed + 1000 + k
        )

        p_mid = vectorized_spearman_power(
            mid,
            residual_sd,
            n_rep,
            rng,
            N_SIMS
        )

        if p_mid >= target_power:
            hi = mid
        else:
            lo = mid

    # --------------------------------------------------------------
    # 3. Final power evaluation at reported MDE
    # --------------------------------------------------------------

    rng = np.random.default_rng(
        seed + 5000
    )

    final_power = vectorized_spearman_power(
        hi,
        residual_sd,
        n_rep,
        rng,
        N_SIMS
    )

    return float(hi), float(final_power)


# --------------------------------------------------------------
# Run MDE calculation for all four endpoints
# --------------------------------------------------------------

mde_rows = []

for row in observed_df.itertuples(index=False):

    ep_index = ENDPOINTS.index(
        row.endpoint
    )

    print(
        f"Computing MDE for {row.endpoint} ..."
    )

    mde, achieved_power = estimate_mde_fast(
        row.residual_sd,
        N_REDUCED,
        TARGET_POWER,
        SEED + ep_index * 10000
    )

    mde_rows.append({
        "endpoint": row.endpoint,
        "n_replicates_per_rho": N_REDUCED,
        "residual_sd": row.residual_sd,
        "MDE_absolute_linear_slope_at_80pct_power": mde,
        "simulated_power_at_MDE": achieved_power,
        "target_power": TARGET_POWER,
        "test": (
            "Spearman correlation with "
            "parametric p-value"
        ),
    })


mde_df = pd.DataFrame(
    mde_rows
)

display(mde_df)

Computing MDE for delta_z_hair ...
Computing MDE for D ...
Computing MDE for D_AUROC ...
Computing MDE for xai_enrichment ...


,endpoint,n_replicates_per_rho,residual_sd,MDE_absolute_linear_slope_at_80pct_power,simulated_power_at_MDE,target_power,test
0,delta_z_hair,3,0.513180,1.094517,0.79280,0.8,Spearman correlation with parametric p-value
1,D,3,0.012008,0.025610,0.79860,0.8,Spearman correlation with parametric p-value
2,D_AUROC,3,0.018914,0.040341,0.79515,0.8,Spearman correlation with parametric p-value
3,xai_enrichment,3,0.812837,1.736805,0.79925,0.8,Spearman correlation with parametric p-value


In [9]:
# CELL 6 — decision-boundary check

boundary = power_df[
    (power_df["design"] == "n=5 primary") &
    (power_df["boundary_flag_75_85pct"])
].copy()

if len(boundary):
    print("BOUNDARY FLAG: one or more n=5 power estimates are within ±5 percentage points of 80%.")
    print("Run the true permutation-Spearman confirmation helper in Cell 7 for those endpoints.")
    display(boundary[["endpoint", "simulated_power", "target_power"]])
else:
    print("No n=5 power estimate lies within ±5 percentage points of the 80% decision boundary.")
    print("The parametric-Spearman approximation is therefore not decision-critical for this design-sensitivity analysis.")

print()
print("Primary-test alignment:")
print("- Locked primary inference: permutation-Spearman.")
print("- T1.7 simulation: parametric Spearman p-value for computational speed.")
print("- T1.7 does not modify, replace, or reopen the locked primary inference.")


No n=5 power estimate lies within ±5 percentage points of the 80% decision boundary.
The parametric-Spearman approximation is therefore not decision-critical for this design-sensitivity analysis.

Primary-test alignment:
- Locked primary inference: permutation-Spearman.
- T1.7 simulation: parametric Spearman p-value for computational speed.
- T1.7 does not modify, replace, or reopen the locked primary inference.


In [10]:
# CELL 7 — true permutation-Spearman confirmation helper

def permutation_spearman_pvalue(x, y, n_perm=N_CONFIRM, rng=None):
    """Two-sided Monte Carlo permutation p-value for Spearman rho."""
    if rng is None:
        rng = np.random.default_rng(SEED)

    observed_stat = float(stats.spearmanr(x, y).statistic)
    null = np.empty(n_perm, dtype=float)

    for i in range(n_perm):
        null[i] = float(stats.spearmanr(x, rng.permutation(y)).statistic)

    return (np.count_nonzero(np.abs(null) >= abs(observed_stat)) + 1) / (n_perm + 1)


print(
    "Helper defined. It is intentionally not run automatically. "
    "Use it only if Cell 6 flags an n=5 power estimate near 80%."
)


Helper defined. It is intentionally not run automatically. Use it only if Cell 6 flags an n=5 power estimate near 80%.


In [11]:
# CELL 8 — save auditable evidence package

OUT = Path("/kaggle/working/t1_7_results")
OUT.mkdir(parents=True, exist_ok=True)

observed_df.to_csv(
    OUT / "T1_7_observed_spearman_effect_noise.csv",
    index=False
)

power_df.to_csv(
    OUT / "T1_7_retrospective_power_n5_n3_spearman.csv",
    index=False
)

mde_df.to_csv(
    OUT / "T1_7_MDE_n3_80pct_power_spearman.csv",
    index=False
)

summary = power_df.merge(
    mde_df[["endpoint", "MDE_absolute_linear_slope_at_80pct_power"]],
    on="endpoint",
    how="left"
).merge(
    observed_df[["endpoint", "spearman_rho", "parametric_spearman_p"]],
    on="endpoint",
    how="left"
)

summary.to_csv(OUT / "T1_7_summary.csv", index=False)

manifest = {
    "analysis": "T1.7 retrospective power / replicate-count justification",
    "input": str(CSV_PATH),
    "n_models_primary": 30,
    "rho_levels": RHO_LEVELS.tolist(),
    "replicates_primary": N_PRIMARY,
    "replicates_reduced": N_REDUCED,
    "alpha": ALPHA,
    "target_power": TARGET_POWER,
    "boundary_tolerance": BOUNDARY_TOL,
    "n_simulations": N_SIMS,
    "seed": SEED,
    "test": "Spearman correlation with parametric p-value",
    "primary_test": "Locked primary uses permutation-Spearman",
    "primary_test_approximation": "T1.7 uses parametric Spearman p-value for simulation speed",
    "true_permutation_confirmation": "Use only if n=5 simulated power is within ±5 percentage points of 80%",
    "interpretation": "Retrospective design sensitivity; MDE conditional on residual variability estimated from the completed primary experiment.",
}

(OUT / "T1_7_AUDIT_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2)
)

print("Saved T1.7 evidence package to:", OUT)
for p in sorted(OUT.iterdir()):
    print(" -", p.name)


Saved T1.7 evidence package to: /kaggle/working/t1_7_results
 - T1_7_AUDIT_MANIFEST.json
 - T1_7_MDE_n3_80pct_power_spearman.csv
 - T1_7_observed_spearman_effect_noise.csv
 - T1_7_retrospective_power_n5_n3_spearman.csv
 - T1_7_summary.csv


## Interpretation guardrails

1. Do **not** describe this as prospective power or as proof that the experiment was adequately powered before data collection.
2. Report it as **retrospective design sensitivity** based on the observed residual variability.
3. The n=5 calculation evaluates the completed primary replicate count.
4. The n=3 MDE quantifies the smallest linear rho-associated effect detectable with approximately 80% simulated power under the observed noise scale.
5. The simulation uses the **parametric Spearman p-value**, not the full primary permutation test.
6. If n=5 power is near 80%, perform the true permutation-Spearman confirmation for that endpoint.
7. T1.7 does not modify the locked primary H1/H2 inference or Holm correction.
